# AI Fraud Sentinel — identifying and prioritizing suspicious transactions

**Objective.** Use labeled 2025 transactions to develop and evaluate a fraud classifier, then create a prioritized review queue for the January 2026 records.

**Approach.** Binary classification with CatBoost, time-based validation, and a risk score designed to support human review.

> **How to read the January results:** the January 2026 file does not include `flag_fraude`. The model therefore produces risk estimates; it cannot confirm which transactions were actually fraudulent. Model performance is measured only on the historical holdout period.

## How to run the notebook
1. Place this notebook and both original CSV files in the same folder.
2. Install the dependencies with `python -m pip install -r requirements.txt`.
3. Open the notebook and run the cells from top to bottom, or select **Run All**.
4. The notebook creates `transacoes_jan2026_classificadas.csv` when it finishes.

All 44,451 original January rows are retained, with three prediction fields added. Original column names are left unchanged to preserve compatibility with the supplied data.

## Executive summary of the reference run

**Model:** CatBoost Classifier. The 2025 history was split chronologically: January-September for training, October-November for threshold calibration, and December for an independent test.

| Result | Value |
|---|---:|
| ROC-AUC on the December test set | **0.910** |
| Average Precision on the test set | **0.609** |
| Precision / Recall / F1 | **62.8% / 57.6% / 0.601** |
| Threshold selected during calibration | **0.293** |
| January 2026 transactions scored | **44,451** |
| Transactions flagged for review | **2,122 (4.78%)** |
| High risk / Review / Low risk | **1,025 / 1,097 / 42,329** |

These figures come from the reference run already completed on the case files. The notebook contains the steps to reproduce the evaluation and regenerate the January CSV. Since January has no ground-truth labels, accuracy metrics cannot be calculated for that month.

## 0. Environment setup

Required libraries are listed in `requirements.txt`. If they are not installed, run this command in the terminal with your virtual environment activated:

`python -m pip install -r requirements.txt`

Then open Jupyter Notebook and run the cells in order. On computers with corporate security policies, DLL blocks should be handled with IT support; do not disable security controls.

In [ ]:
# %pip install pandas numpy matplotlib scikit-learn catboost

## 1. Importing the libraries

- **pandas and numpy:** reading, organizing, and preparing the data.
- **matplotlib:** creating charts.
- **scikit-learn:** evaluation metrics and threshold selection.
- **CatBoost:** a classification algorithm with native support for categorical features.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 2. Loading the data

The notebook loads both CSV files in full. The 2025 history contains the `flag_fraude` target used for training and evaluation. The January 2026 file has no such field and is used only to generate predictions.

In [ ]:
ARQUIVO_HISTORICO = Path("transacoes_historico_2025.csv")
ARQUIVO_JANEIRO = Path("transacoes_jan2026.csv")

for arquivo in [ARQUIVO_HISTORICO, ARQUIVO_JANEIRO]:
    if not arquivo.exists():
        raise FileNotFoundError(
            f"Could not find {arquivo}. Place both CSV files in the same folder as this notebook."
        )

historico = pd.read_csv(ARQUIVO_HISTORICO, low_memory=False)
janeiro = pd.read_csv(ARQUIVO_JANEIRO, low_memory=False)

print(f"Histórico 2025: {historico.shape[0]:,} linhas x {historico.shape[1]} colunas")
print(f"Janeiro 2026:   {janeiro.shape[0]:,} linhas x {janeiro.shape[1]} colunas")
print("\nHistorical data columns:")
print(historico.columns.tolist())
print("\nJanuary data columns:")
print(janeiro.columns.tolist())

## 3. Data quality and integrity checks

Before modeling, we check record counts, columns, missing values, duplicate rows, invalid dates, and the observed date range. We also compare the structure of both files so that records or fields are not silently dropped during processing.

Gaps between transaction timestamps are counted separately. They are not missing values within a row, and no artificial transactions are created to fill them.

In [ ]:
def auditar_base(df, nome, tem_alvo=False):
    datas = pd.to_datetime(df["datahora_transacao"], errors="coerce")
    diferencas = datas.sort_values().diff().dropna()
    lacunas = diferencas[diferencas > pd.Timedelta(minutes=1)]
    minutos_sem_registro = int(((lacunas / pd.Timedelta(minutes=1)) - 1).sum())
    resumo = {
        "dataset": nome,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "invalid_dates": int(datas.isna().sum()),
        "duplicate_timestamps": int(datas.duplicated().sum()),
        "gaps_over_one_minute": int(len(lacunas)),
        "unrecorded_minutes_in_observed_interval": minutos_sem_registro,
        "first_timestamp": str(datas.min()),
        "last_timestamp": str(datas.max()),
    }
    if tem_alvo:
        resumo["fraud_cases"] = int(df["flag_fraude"].astype(bool).sum())
        resumo["fraud_rate"] = float(df["flag_fraude"].astype(bool).mean())
    return resumo

auditoria = pd.DataFrame([
    auditar_base(historico, "Historical data 2025", tem_alvo=True),
    auditar_base(janeiro, "January 2026", tem_alvo=False),
])
display(auditoria)

# Verificações explícitas de integridade
assert historico.shape == (523525, 15), "Unexpected number of rows in the historical data. Check the source file."
assert janeiro.shape == (44451, 14), "Unexpected number of rows in the January data. Check the source file."
assert historico.isna().sum().sum() == 0
assert janeiro.isna().sum().sum() == 0
assert historico.duplicated().sum() == 0
assert janeiro.duplicated().sum() == 0
assert pd.to_datetime(historico["datahora_transacao"], errors="coerce").notna().all()
assert pd.to_datetime(janeiro["datahora_transacao"], errors="coerce").notna().all()
assert "flag_fraude" in historico.columns and "flag_fraude" not in janeiro.columns
assert set(janeiro.columns).issubset(set(historico.columns))
print("\nAuditoria concluída: todas as linhas e colunas esperadas foram carregadas.")

## 4. Exploratory analysis: understanding the target

Fraud represents a small share of all transactions. For that reason, **accuracy alone can be misleading**: a model that labels every transaction as legitimate would achieve high accuracy while identifying no fraud at all. We also look at precision, recall, F1, and Average Precision, which provide a more useful view of performance in this setting.

In [ ]:
historico["datahora_dt"] = pd.to_datetime(historico["datahora_transacao"])
historico["hora_eda"] = historico["datahora_dt"].dt.hour

fraude_por_classe = historico["flag_fraude"].value_counts().rename(index={False: "Legitimate", True: "Fraud"})
print("Target distribution:")
display(fraude_por_classe.to_frame("transactions"))
print(f"Historical fraud rate: {historico['flag_fraude'].mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fraude_por_classe.plot(kind="bar", ax=axes[0], title="Transaction distribution", rot=0)
axes[0].set_ylabel("Transactions")
axes[0].set_xlabel("")

fraude_hora = historico.groupby("hora_eda")["flag_fraude"].mean()
fraude_hora.plot(kind="bar", ax=axes[1], title="Historical fraud rate by hour", color="#247C8A")
axes[1].set_ylabel("Fraud rate")
axes[1].set_xlabel("Hour of day")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.show()

print("Historical fraud rate by device status:")
display(historico.groupby("is_new_device")["flag_fraude"].agg(["count", "sum", "mean"]).rename(index={False: "Known device", True: "New device"}))

### What the data shows

- The 2025 history contains **20,485 labeled fraud cases across 523,525 transactions (3.91%)**.
- The observed fraud rate was **12.9%** for new devices, compared with **2.1%** for known devices. This is an association in the supplied data, not evidence of causation.
- Transaction hour shows a particularly strong pattern. Some hours have no labeled fraud, while others show a concentration of cases. This should be reviewed with the business team before it is used to guide operational rules.

## 5. Feature engineering

The transformations are intentionally simple and traceable:

- `datahora_transacao` is converted into hour, day of week, day of month, month, and weekend indicator features.
- `nome_cliente` and `codigo_postal` are excluded because their high cardinality could encourage the model to memorize identifiers rather than learn patterns that generalize.
- The original timestamp is dropped after the time-based features are created.
- The remaining columns are considered as candidate predictors by the model.

The source datasets are not modified. Feature engineering is performed on DataFrame copies.

In [ ]:
def criar_variaveis_modelo(df):
    dados = df.copy()
    datas = pd.to_datetime(dados["datahora_transacao"], errors="raise")
    dados["hora"] = datas.dt.hour
    dados["dia_semana"] = datas.dt.dayofweek
    dados["dia_mes"] = datas.dt.day
    dados["mes"] = datas.dt.month
    dados["fim_de_semana"] = (datas.dt.dayofweek >= 5).astype("int8")
    dados = dados.drop(columns=["datahora_transacao", "nome_cliente", "codigo_postal"])
    return dados

base_modelo = criar_variaveis_modelo(historico)
X = base_modelo.drop(columns=["flag_fraude"])
y = base_modelo["flag_fraude"].astype(int)

# CatBoost receives categorical features as text.
colunas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
for coluna in colunas_categoricas:
    X[coluna] = X[coluna].astype(str)

print("Features used:", X.shape[1])
print("Categorical features:", colunas_categoricas)
print("Numeric features:", [c for c in X.columns if c not in colunas_categoricas])

## 6. Time-based train, calibration, and test split

Rather than shuffling transactions, we use consecutive time periods. This better reflects the intended use: learn from past activity and score transactions from a later period.

| Period | Purpose | Rows |
|---|---|---:|
| Jan-Sep 2025 | Training | 391,566 |
| Oct-Nov 2025 | Threshold calibration | 87,508 |
| Dec 2025 | Final test | 44,451 |

December is kept out of both model fitting and threshold selection, providing an evaluation on a later month.

In [ ]:
datas_historico = pd.to_datetime(historico["datahora_transacao"])
mascara_treino = datas_historico < "2025-10-01"
mascara_calibracao = (datas_historico >= "2025-10-01") & (datas_historico < "2025-12-01")
mascara_teste = datas_historico >= "2025-12-01"

X_treino, y_treino = X.loc[mascara_treino], y.loc[mascara_treino]
X_calibracao, y_calibracao = X.loc[mascara_calibracao], y.loc[mascara_calibracao]
X_teste, y_teste = X.loc[mascara_teste], y.loc[mascara_teste]

print("Training:", X_treino.shape[0], "| taxa de fraude:", f"{y_treino.mean():.2%}")
print("Calibration:", X_calibracao.shape[0], "| taxa de fraude:", f"{y_calibracao.mean():.2%}")
print("Test:", X_teste.shape[0], "| taxa de fraude:", f"{y_teste.mean():.2%}")

## 7. Model training

We use **CatBoost**, a gradient-boosted decision tree algorithm that can handle categorical features directly. Training uses early stopping: if performance on the calibration data stops improving, the process is halted rather than continuing to fit without an observed gain.

In [ ]:
modelo = CatBoostClassifier(
    iterations=600,
    depth=7,
    learning_rate=0.07,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=100,
    thread_count=4,
    l2_leaf_reg=5,
)

modelo.fit(
    X_treino,
    y_treino,
    cat_features=colunas_categoricas,
    eval_set=(X_calibracao, y_calibracao),
    early_stopping_rounds=60,
)

print("Best iteration:", modelo.get_best_iteration())
modelo.save_model("modelo_catboost_fraude.cbm")
print("Model saved to modelo_catboost_fraude.cbm")

## 8. Selecting the decision threshold

The model produces a score between 0 and 1. A threshold is needed to turn that score into a flag. Because the case does not provide the financial cost of missed fraud or the operational impact of unnecessary alerts, we use the **highest F1 score on the calibration set** as an initial reference, balancing precision and recall.

This value should not be treated as a permanent business rule. Before operational use, the fraud prevention team should review the threshold against the actual costs of each type of error.

In [ ]:
scores_calibracao = modelo.predict_proba(X_calibracao)[:, 1]
precisoes, recalls, limiares = precision_recall_curve(y_calibracao, scores_calibracao)
f1_por_limiar = 2 * precisoes * recalls / (precisoes + recalls + 1e-12)
indice_melhor = int(np.argmax(f1_por_limiar[:-1]))
limiar_decisao = float(limiares[indice_melhor])

print(f"Limiar selecionado na calibração: {limiar_decisao:.4f}")
print(f"Precisão na calibração: {precisoes[indice_melhor]:.2%}")
print(f"Recall na calibração: {recalls[indice_melhor]:.2%}")
print(f"F1 na calibração: {f1_por_limiar[indice_melhor]:.3f}")

## 9. Final evaluation on December 2025

The metrics below are calculated on December, which was not used for model training or threshold selection. Average Precision is useful for evaluating ranking quality on imbalanced datasets and should be read alongside the fraud prevalence in the test set.

In [ ]:
scores_teste = modelo.predict_proba(X_teste)[:, 1]
predicoes_teste = (scores_teste >= limiar_decisao).astype(int)

roc_auc = roc_auc_score(y_teste, scores_teste)
average_precision = average_precision_score(y_teste, scores_teste)
precisao = precision_score(y_teste, predicoes_teste, zero_division=0)
recall = recall_score(y_teste, predicoes_teste, zero_division=0)
f1 = f1_score(y_teste, predicoes_teste, zero_division=0)
acuracia = accuracy_score(y_teste, predicoes_teste)
matriz = confusion_matrix(y_teste, predicoes_teste)
tn, fp, fn, tp = matriz.ravel()

metricas_teste = pd.DataFrame({
    "Metric": ["ROC-AUC", "Average Precision", "Precision", "Recall", "F1", "Accuracy"],
    "Result": [roc_auc, average_precision, precisao, recall, f1, acuracia],
})
display(metricas_teste.style.format({"Result": "{:.3f}"}))

print(f"\nConfusion matrix: TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}")
print(f"Flagged transactions: {int(predicoes_teste.sum()):,} of {len(y_teste):,} ({predicoes_teste.mean():.2%})")
print(f"No-skill baseline (Average Precision = prevalence): {y_teste.mean():.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=["Legitimate", "Fraud"])
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format=",")
ax.set_title("Confusion matrix - December 2025")
plt.tight_layout()
plt.show()

### Reading the confusion matrix

- **Precision:** among the transactions flagged by the model, how many were fraud in the test set.
- **Recall:** among the fraud cases in the test set, how many were identified.
- **False positives:** legitimate transactions flagged for review.
- **False negatives:** fraud cases that were not flagged.

The goal is to organize the review workload, not to block customers automatically. Any blocking decision requires additional context, business rules, and operational validation.

## 10. Features with the greatest influence

Feature importance summarizes how much each field contributed to the model's decisions across the evaluated data. It does not establish a causal relationship. The high importance assigned to transaction hour, in particular, deserves careful monitoring.

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importancias.head(10).to_frame("relative_importance"))

fig, ax = plt.subplots(figsize=(9, 5))
importancias.head(10).sort_values().plot(kind="barh", ax=ax, color="#247C8A")
ax.set_title("Top 10 feature importance")
ax.set_xlabel("Relative feature importance")
plt.tight_layout()
plt.show()

## 11. Scoring January 2026

The model trained on January-September 2025 is used to calculate a score for every January 2026 transaction. The calibrated threshold identifies records for review; a second threshold (0.60) highlights the highest-priority cases.

All 44,451 original rows are retained. The output adds three fields: a score, a binary flag, and a risk band. Risk labels in the CSV remain in Portuguese to preserve the original data convention.

In [ ]:
X_janeiro = criar_variaveis_modelo(janeiro)
X_janeiro = X_janeiro.reindex(columns=X_treino.columns)
for coluna in colunas_categoricas:
    X_janeiro[coluna] = X_janeiro[coluna].astype(str)

scores_janeiro = modelo.predict_proba(X_janeiro)[:, 1]
limiar_alto_risco = 0.60  # illustrative operational cutoff; validate with the business team

resultado_janeiro = janeiro.copy()
resultado_janeiro["score_risco_fraude"] = scores_janeiro
resultado_janeiro["flag_fraude_prevista"] = scores_janeiro >= limiar_decisao
resultado_janeiro["nivel_risco"] = np.select(
    [scores_janeiro >= limiar_alto_risco, scores_janeiro >= limiar_decisao],
    ["Alto", "Atenção"],
    default="Baixo",
)

ARQUIVO_SAIDA = Path("transacoes_jan2026_classificadas.csv")
resultado_janeiro.to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8-sig")

resumo_risco = resultado_janeiro["nivel_risco"].value_counts().reindex(["Alto", "Atenção", "Baixo"], fill_value=0)
print("January 2026 risk breakdown:")
display(resumo_risco.rename(index={"Alto": "High", "Atenção": "Review", "Baixo": "Low"}).to_frame("Transactions"))
print(f"\nTotal classificado: {len(resultado_janeiro):,}")
print(f"Predicted alerts: {resultado_janeiro['flag_fraude_prevista'].sum():,} ({resultado_janeiro['flag_fraude_prevista'].mean():.2%})")
print(f"Output saved to: {ARQUIVO_SAIDA.resolve()}")

## 12. Final checks: validating the output

The last step reads the generated CSV and compares its structure with the original January file. We check the row count, confirm that all 14 source columns are present, and verify the three added prediction fields.

The assertions below stop execution if any expected condition is not met.

In [ ]:
saida_releitura = pd.read_csv(ARQUIVO_SAIDA, low_memory=False)
colunas_originais_preservadas = all(c in saida_releitura.columns for c in janeiro.columns)

assert len(saida_releitura) == len(janeiro)
assert colunas_originais_preservadas
assert saida_releitura["score_risco_fraude"].between(0, 1).all()
assert saida_releitura["nivel_risco"].notna().all()
assert saida_releitura["flag_fraude_prevista"].notna().all()

print("Input rows:", f"{len(janeiro):,}")
print("Output rows:", f"{len(saida_releitura):,}")
print("Original columns preserved:", colunas_originais_preservadas)
print("Added columns:", [c for c in saida_releitura.columns if c not in janeiro.columns])
print("Duplicate rows in output:", saida_releitura.duplicated().sum())
print("Missing values in output:", saida_releitura.isna().sum().sum())

assert len(saida_releitura) == 44451
assert len(saida_releitura.columns) == 17
print("\nVALIDAÇÃO CONCLUÍDA: nenhuma linha da base de janeiro foi perdida.")

## 13. Key findings, limitations, and next steps

### Key findings
- The dataset is imbalanced: 3.91% of the 2025 transactions were labeled as fraud.
- `hora` was the most influential feature (49.5%), followed by device type and merchant type.
- New devices had a higher historical fraud rate, but this difference should be monitored in future samples.
- The December test results reinforce the need to monitor changes in behavior over time.

### Limitations
- January 2026 has no ground-truth labels, so precision, recall, and accuracy cannot be measured for that month.
- The score is a model estimate, not a calibrated probability or confirmation of fraud.
- The threshold was selected using F1 because no business cost matrix was provided.
- The unusually strong time-of-day pattern should be investigated before any automated decision is considered.
- `nome_cliente` and `codigo_postal` were excluded to reduce the risk of memorizing identifiers.

### Recommended next steps
1. Review a sample of alerts with analysts and estimate the cost of false positives and false negatives.
2. Monitor alert volume, model performance, data quality, and changes in patterns (*data drift*).
3. Recalibrate the threshold and retrain when new confirmed labels become available.
4. Evaluate transaction-level explanations (for example, SHAP) before operational adoption.

**Conclusion.** The model shows potential to help prioritize suspicious transactions. The next step is to validate it with the responsible team and keep human review as part of the decision.

---
**Output file:** `transacoes_jan2026_classificadas.csv` (44,451 rows, 17 columns).
**Model:** CatBoost Classifier, using a time-based split and a threshold selected on the calibration set.